# Глава 5. Переобучение и валидация

Код к главе 5 книги 1: пример Рунге, разложение ошибки на смещение и разброс, схемы
валидации, метрики качества регрессии, информационные критерии и подбор гиперпараметров
(листинг 5.1<!-- lst:Выбор степени -->).

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch05_overfitting.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.polynomial import legendre as leg

from mlbook import latency

## Пример Рунге

$f(x) = 1/(1 + 25x^2)$ в 15 равноотстоящих точках отрезка $[-2;\ 2]$ — обучающая выборка;
контрольная — 14 точек посередине между ними. Многочлены степеней 0–14 строим МНК.
Чтобы не мешала плохая обусловленность (гл. 3), берём базис многочленов Лежандра: сами
многочлены те же, что в базисе $1, x, x^2, \ldots$, но система решается точно.

In [ ]:
f = lambda x: 1 / (1 + 25 * x ** 2)
x = np.linspace(-2, 2, 15)
xt = (x[:-1] + x[1:]) / 2


def fit(d):
    return np.linalg.lstsq(leg.legvander(x / 2, d), f(x), rcond=None)[0]


def pred(c, z):
    return leg.legvander(z / 2, len(c) - 1) @ c


degrees = np.arange(15)
q_train = np.array([np.mean((pred(fit(d), x) - f(x)) ** 2) for d in degrees])
q_test = np.array([np.mean((pred(fit(d), xt) - f(xt)) ** 2) for d in degrees])
for d in (2, 8, 14):
    print(f"d = {d:2}: обучение {q_train[d]:.3g}, контроль {q_test[d]:.3g}")

In [ ]:
xs = np.linspace(-2, 2, 600)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
a.plot(xs, f(xs), color="0.6", linewidth=2, label="$f(x)$")
for d in (2, 8, 14):
    a.plot(xs, pred(fit(d), xs), label=f"степень {d}")
a.scatter(x, f(x), color="black", zorder=3, s=12, label="обучение")
a.scatter(xt, f(xt), marker="x", color="black", zorder=3, label="контроль")
a.set_ylim(-0.6, 1.6)
a.legend(fontsize=8)
a.set_title("а) аппроксимации")
b.plot(degrees, q_test, "o-", label="контроль")
b.plot(degrees[:-1], q_train[:-1], "s-", label="обучение")   # при d = 14 ошибка ≈ 0
b.set_yscale("log")
b.set_xlabel("степень многочлена $d$")
b.set_ylabel("$Q$")
b.set_title("б) ошибка от степени")
b.legend()
plt.show()

Ошибки не меняются при переходе от чётной степени к нечётной: $f$ чётная, коэффициенты
при нечётных степенях нулевые.

In [ ]:
print(np.allclose(q_train[0:14:2], q_train[1:15:2]), fit(8)[1::2].round(12))

## Смещение и разброс

$\varphi(x) = \sin 2\pi x$, 15 равноотстоящих точек отрезка $[0;\ 1]$, шум $\sigma = 0{,}3$.
Обучим многочлены на 4000 выборках и вычислим слагаемые разложения, усреднив их по
отрезку (табл. 5.1<!-- tbl:bias-variance -->). Выборки случайные, поэтому третий знак может отличаться от таблицы.

In [ ]:
rng = np.random.default_rng(0)
phi = lambda x: np.sin(2 * np.pi * x)
L, SIGMA, R = 15, 0.3, 4000
xb = np.linspace(0, 1, L)
grid = np.linspace(0, 1, 400)

print("степень  смещение²  разброс   шум   ожидаемая ошибка")
for d in (1, 3, 11):
    V, Vg = leg.legvander(2 * xb - 1, d), leg.legvander(2 * grid - 1, d)
    Y = phi(xb) + rng.normal(0, SIGMA, (R, L))                 # R выборок сразу
    G = (np.linalg.lstsq(V, Y.T, rcond=None)[0].T) @ Vg.T      # прогнозы R моделей на сетке
    bias2 = np.mean((G.mean(axis=0) - phi(grid)) ** 2)
    var = np.mean(G.var(axis=0))
    print(f"{d:5}  {bias2:9.3f}  {var:8.3f}  {SIGMA ** 2:5.2f}  {bias2 + var + SIGMA ** 2:8.3f}")

Двадцать моделей степени 1 и 11 (рис. 5.2<!-- fig:bias-variance -->): прямые близки друг к другу, но далеки от
синусоиды; многочлены степени 11 в среднем повторяют синусоиду, но каждый — по-своему.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, d in zip(axes, (1, 11)):
    V, Vg = leg.legvander(2 * xb - 1, d), leg.legvander(2 * grid - 1, d)
    P = [Vg @ np.linalg.lstsq(V, phi(xb) + rng.normal(0, SIGMA, L), rcond=None)[0]
         for _ in range(20)]
    for p in P:
        ax.plot(grid, p, color="C0", linewidth=0.5, alpha=0.5)
    ax.plot(grid, np.mean(P, axis=0), "--", color="C1", label="средняя модель")
    ax.plot(grid, phi(grid), color="black", label=r"$\varphi(x)$")
    ax.set_title(f"степень {d}")
    ax.set_ylim(-2, 2)
axes[0].legend()
plt.show()

## Схемы валидации

Номера объектов в контрольных блоках: $k$-кратная кросс-валидация, стратифицированная
(доли классов в блоках как во всей выборке) и валидация для временного ряда, где контроль
всегда позже обучения.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, TimeSeriesSplit

idx = np.arange(10)
labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
for name, cv in [("KFold", KFold(5)), ("StratifiedKFold", StratifiedKFold(2)),
                 ("TimeSeriesSplit", TimeSeriesSplit(4))]:
    print(name)
    for tr, te in cv.split(idx, labels):
        print("  обучение", tr, " контроль", te)

## Метрики качества регрессии

**$R^2$ на контроле бывает отрицательным.** Модель $g(x) = 1 + 0{,}5x$ из гл. 3 на
обучающих точках и на четырёх новых.

In [ ]:
from sklearn.metrics import r2_score

g = lambda x: 1 + 0.5 * x
x_tr, y_tr = np.array([1, 2, 3]), np.array([1, 3, 2])
x_te, y_te = np.array([0, 1, 2, 4]), np.array([2, 2.5, 1.5, 2])
print("R² на обучении:", r2_score(y_tr, g(x_tr)), " на контроле:", r2_score(y_te, g(x_te)))

**Задача 1.** Ответы 2, 4, 6, прогнозы 3, 4, 8.

In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error)

y_true, y_hat = np.array([2, 4, 6]), np.array([3, 4, 8])
mse = mean_squared_error(y_true, y_hat)
print(f"MSE = {mse:.3f}, RMSE = {np.sqrt(mse):.3f}, MAE = {mean_absolute_error(y_true, y_hat)},",
      f"MAPE = {100 * mean_absolute_percentage_error(y_true, y_hat):.1f} %,",
      f"R² = {r2_score(y_true, y_hat)}")

**Задача 3.** Информационные критерии: $\mathrm{AIC} = \ell \ln \mathrm{MSE} + 2(n + 1)$,
$\mathrm{BIC} = \ell \ln \mathrm{MSE} + (n + 1) \ln \ell$. Меньше — лучше.

In [ ]:
ell = 100
for params, mse in [(3, 2.0), (10, 1.8)]:
    print(f"{params} параметров: AIC = {ell * np.log(mse) + 2 * params:.1f},",
          f"BIC = {ell * np.log(mse) + params * np.log(ell):.1f}")

## Подбор гиперпараметров

Выбор степени многочлена кросс-валидацией на данных о времени отклика (200 измерений,
истинная зависимость — квадратичная) и итоговая оценка на тестовой выборке — листинг 5.1<!-- lst:Выбор степени -->.

In [ ]:
df = latency.load()
X, y = df[["нагрузка"]], df["время"]

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=0)
model = make_pipeline(StandardScaler(), PolynomialFeatures(),
                      LinearRegression())
search = GridSearchCV(
    model, {"polynomialfeatures__degree": range(1, 11)},
    cv=5, scoring="neg_mean_squared_error")
search.fit(X_tr, y_tr)      # кросс-валидация на обучающей части
print(search.best_params_)
# тестовая выборка — один раз, в конце
y_pred = search.predict(X_te)
print(mean_squared_error(y_te, y_pred))

С чем сравнить ошибку на тесте? Данные синтетические, и истинная зависимость известна;
её ошибка на тех же тестовых объектах — предел, лучше которого не сделает никакая модель.
(В среднем он равен дисперсии шума $8^2 = 64$, но на 40 объектах оценка сама случайна.)

In [ ]:
print(mean_squared_error(y_te, latency.response_time(X_te["нагрузка"])))

Кривая ошибки кросс-валидации по степеням:

In [ ]:
cv_mse = -search.cv_results_["mean_test_score"]
cv_std = search.cv_results_["std_test_score"]
for d, m, s in zip(range(1, 11), cv_mse, cv_std):
    print(f"степень {d:2}: MSE на кросс-валидации {m:7.1f} ± {s:.1f}")

Степени от 2 до 10 различаются меньше, чем ошибки на разных блоках: по такой разнице
выбирать нельзя (разд. 5.4<!-- sec:Схемы валидации -->), и разумно взять самую простую из них.